# Parallelization: los especialistas a la vez

Clasificación: **Workflow paralelo.** Vuelos, alojamiento, actividades y transporte buscan opciones a la vez, cada uno con el presupuesto total como referencia. El coordinador ajusta al final.

## Diferencia con chaining

En chaining cada agente sabe "cuánto queda" tras el anterior. Aquí no: cada uno propone lo mejor para su partida sin saber qué gastan los demás. El coordinador final es quien cuadra el total.

Esto funciona porque la búsqueda de opciones es independiente. Lo que depende del presupuesto restante es la decisión final, no la exploración.

## Cómo funciona en CrewAI

CrewAI no tiene un `Process.parallel`. El paralelismo se consigue con `async_execution=True` en cada task independiente. El motor las lanza a la vez. La task final usa `context=[task1, task2, ...]` para esperar a todas antes de ejecutarse.

```python
vuelos_task = Task(..., async_execution=True)
alojamiento_task = Task(..., async_execution=True)

combinar_task = Task(
    ...,
    context=[vuelos_task, alojamiento_task],  # espera a ambas
)
```

In [1]:
!uv pip install -r requirements.txt --quiet

In [2]:
from dotenv import load_dotenv
import nest_asyncio

load_dotenv()
nest_asyncio.apply()

## Ejecución en paralelo

Reutilizamos los agentes y la config de tasks de `ViajesCrew`. Lo único nuevo es `async_execution=True` en las 4 tasks independientes y un agente coordinador que las combina al final.

In [3]:
from crewai import Task, Crew, Process, Agent
from viajes_crew import ViajesCrew

inputs = {"destino": "Islandia", "dias": 5, "personas": 2, "presupuesto": 2200}

base_crew = ViajesCrew()

# Tasks paralelas: cada una busca opciones sin depender de las demás
vuelos_task = Task(
    description="Propon 2-3 opciones de vuelo a {destino} para {personas} personas y {dias} dias. Presupuesto total del viaje: {presupuesto} EUR.",
    expected_output="2-3 opciones de vuelo con precio aproximado y fechas.",
    agent=base_crew.vuelos(),
    async_execution=True,
)
alojamiento_task = Task(
    description="Propon 2 opciones de alojamiento en {destino} para {personas} personas y {dias} noches. Presupuesto total del viaje: {presupuesto} EUR.",
    expected_output="2 opciones de alojamiento con precio por noche y zona.",
    agent=base_crew.alojamiento(),
    async_execution=True,
)
actividades_task = Task(
    description="Propon actividades para {dias} dias en {destino} para {personas} personas. Presupuesto total del viaje: {presupuesto} EUR.",
    expected_output="Actividades dia por dia con coste estimado.",
    agent=base_crew.actividades(),
    async_execution=True,
)
transporte_task = Task(
    description="Propon opciones de transporte para moverse en {destino} durante {dias} dias. Presupuesto total del viaje: {presupuesto} EUR.",
    expected_output="2-3 opciones de transporte con precio y recomendacion.",
    agent=base_crew.transporte(),
    async_execution=True,
)

# Task final: espera a las 4, ajusta presupuesto
coordinador = Agent(
    role="Coordinador de Itinerario",
    goal="Combinar propuestas y ajustar para que el total no supere el presupuesto",
    backstory="Ensamblas piezas sueltas en un itinerario coherente. Si la suma supera el presupuesto, eliges las opciones mas economicas.",
)
combinar_task = Task(
    description=(
        "Con las 4 propuestas anteriores, elige la mejor combinacion que encaje en {presupuesto} EUR total.\n"
        "Ensambla un itinerario dia a dia para {destino}, {dias} dias, {personas} personas.\n"
        "Si la suma supera el presupuesto, ajusta eligiendo opciones mas baratas."
    ),
    expected_output="Itinerario final con vuelos, alojamiento, actividades, transporte, coste por partida y total.",
    agent=coordinador,
    context=[vuelos_task, alojamiento_task, actividades_task, transporte_task],
)

crew = Crew(
    agents=[base_crew.vuelos(), base_crew.alojamiento(), base_crew.actividades(), base_crew.transporte(), coordinador],
    tasks=[vuelos_task, alojamiento_task, actividades_task, transporte_task, combinar_task],
    process=Process.sequential,
    verbose=True,
)

result = await crew.kickoff_async(inputs=inputs)
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: dabba1c7-5c28-4c60-a523-cc6fb675559b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Propon 2-3 opciones de vuelo a Islandia para 2 personas y 5 dias. Presupuesto total del viaje: 2200      │
│  EUR.                                                                                                           │
│  ID: 7d1f76e7-23f3-4bae-a192-37f312d06e66                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Propon 2 opciones de alojamiento en Islandia para 2 personas y 5 noches. Presupuesto total del viaje:    │
│  2200 EUR.                                                                                                      │
│  ID: bf677fb0-fe05-417e-9e2d-e1d93597b0b0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Propon actividades para 5 dias en Islandia para 2 personas. Presupuesto total del viaje: 2200 EUR.       │
│  ID: 1a505fe6-ba65-4d31-8222-f3b62d4e7812                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Propon opciones de transporte para moverse en Islandia durante 5 dias. Presupuesto total del viaje:      │
│  2200 EUR.                                                                                                      │
│  ID: 87af4458-eed4-40ae-9f67-dd6aaceb1b66                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Coordinador de Itinerario                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Aquí tienes el itinerario final detallado para 5 días en Islandia para 2 personas, combinado con vuelos,       │
│  alojamiento, transporte y actividades, ajustado para no superar el presupuesto total de 2200 EUR.              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Itinerario Final Islandia 5 Días para 2 Personas                                                            │
│                                                                                                                 │
│  ### Elección de Vuelos:                                                                                        │
│  **Opción 2: Vuelo con escala desde Barcelona a Reikiavik con Norwegian/EasyJet**                               │
│  - Ida: 11 de junio 2024                                                                                        │
│  - Regreso: 16 de junio 2024                                                                                    │
│  - Precio aproximado total para 2 personas: 440 EUR                                                             │
│  - Ventaja: vuelo económico, menor gasto inicial, escala corta tolerable.                                       │
│                                                                                                                 │
│  ### Alojamiento:                                                                                               │
│  **Opción 1 – Airbnb: Apartamento moderno en Reikiavik (Miðborg) 5 noches**                                     │
│  - Precio: 150 EUR/noche × 5 noches = 750 EUR                                                                   │
│  - Apartamento céntrico, 2 dormitorios, cocina equipada para preparar algunas comidas y ahorrar en              │
│  restaurantes.                                                                                                  │
│                                                                                                                 │
│  ### Transporte:                                                                                                │
│  **Alquiler de coche pequeño 5 días**                                                                           │
│  - Precio estimado alquiler económico: 50 EUR por día × 5 = 250 EUR                                             │
│  - Gasolina estimada (aprox. 300-400 km total): 60 EUR                                                          │
│  - Total transporte: 310 EUR                                                                                    │
│  - Ventaja: máxima flexibilidad para excursiones a Círculo Dorado, Costa Sur, Jökulsárlón y Blue Lagoon sin     │
│  depender de tours caros.                                                                                       │
│                                                                                                                 │
│  ### Plan de actividades y costes por día                                                                       │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Con las 4 propuestas anteriores, elige la mejor combinacion que encaje en 2200 EUR total.                │
│  Ensambla un itinerario dia a dia para Islandia, 5 dias, 2 personas.                                            │
│  Si la suma supera el presupuesto, ajusta eligiendo opciones mas baratas.                                       │
│  Agent: Coordinador de Itinerario                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: dabba1c7-5c28-4c60-a523-cc6fb675559b                                                                       │
│  Final Output: Aquí tienes el itinerario final detallado para 5 días en Islandia para 2 personas, combinado     │
│  con vuelos, alojamiento, transporte y actividades, ajustado para no superar el presupuesto total de 2200 EUR.  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Itinerario Final Islandia 5 Días para 2 Personas                                                            │
│                                                                                                                 │
│  ### Elección de Vuelos:                                                                                        │
│  **Opción 2: Vuelo con escala desde Barcelona a Reikiavik con Norwegian/EasyJet**                               │
│  - Ida: 11 de junio 2024                                                                                        │
│  - Regreso: 16 de junio 2024                                                                                    │
│  - Precio aproximado total para 2 personas: 440 EUR                                                             │
│  - Ventaja: vuelo económico, menor gasto inicial, escala corta tolerable.                                       │
│                                                                                                                 │
│  ### Alojamiento:                                                                                               │
│  **Opción 1 – Airbnb: Apartamento moderno en Reikiavik (Miðborg) 5 noches**                                     │
│  - Precio: 150 EUR/noche × 5 noches = 750 EUR                                                                   │
│  - Apartamento céntrico, 2 dormitorios, cocina equipada para preparar algunas comidas y ahorrar en              │
│  restaurantes.                                                                                                  │
│                                                                                                                 │
│  ### Transporte:                                                                                                │
│  **Alquiler de coche pequeño 5 días**                                                                           │
│  - Precio estimado alquiler económico: 50 EUR por día × 5 = 250 EUR                                             │
│  - Gasolina estimada (aprox. 300-400 km total): 60 EUR                                                          │
│  - Total transporte: 310 EUR                                                                                    │
│  - Ventaja: máxima flexibilidad para excursiones a Círculo Dorado, Costa Sur, Jökulsárlón y Blue Lagoon sin     │
│  depender de tours caros.                                                                                       │
│                                                                                                                 │
│  ### Plan de actividades y costes por día                                                                       │
│                                                       

Aquí tienes el itinerario final detallado para 5 días en Islandia para 2 personas, combinado con vuelos, alojamiento, transporte y actividades, ajustado para no superar el presupuesto total de 2200 EUR.

---

## Itinerario Final Islandia 5 Días para 2 Personas

### Elección de Vuelos:
**Opción 2: Vuelo con escala desde Barcelona a Reikiavik con Norwegian/EasyJet**  
- Ida: 11 de junio 2024  
- Regreso: 16 de junio 2024  
- Precio aproximado total para 2 personas: 440 EUR  
- Ventaja: vuelo económico, menor gasto inicial, escala corta tolerable.

### Alojamiento:
**Opción 1 – Airbnb: Apartamento moderno en Reikiavik (Miðborg) 5 noches**  
- Precio: 150 EUR/noche × 5 noches = 750 EUR  
- Apartamento céntrico, 2 dormitorios, cocina equipada para preparar algunas comidas y ahorrar en restaurantes.

### Transporte:
**Alquiler de coche pequeño 5 días**  
- Precio estimado alquiler económico: 50 EUR por día × 5 = 250 EUR  
- Gasolina estimada (aprox. 300-400 km total): 60 EUR  
- Total transp

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ 0b596954-38ec-47c1-bba9-258f4faa6e5e                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/trace_batches/0b596954-38ec-47c1-bba9-258 │
│ f4faa6e5e                                                                    │
╰──────────────────────────────────────────────────────────────────────────────╯


## Qué define este patrón

Las 4 búsquedas no se necesitan entre sí. Solo el coordinador final las necesita a todas.

| Chaining | Parallel |
|----------|----------|
| vuelos → alojamiento → actividades → transporte → itinerario | vuelos + alojamiento + actividades + transporte → coordinador |
| Cada paso sabe el presupuesto restante | Cada paso busca con presupuesto total, coordinador ajusta |
| Más lento, más preciso en cada paso | Más rápido, ajuste final centralizado |

El tradeoff: en chaining cada agente optimiza con info exacta del presupuesto restante. En parallel buscan sin esa info pero van 4x más rápido, y el coordinador cuadra al final. Para exploración de opciones, el parallel es mejor.